# Notebook for collecting and filtering of data with Guardian API

API KEY Free here: https://open-platform.theguardian.com/access/

## Imports

In [1]:
import requests
import time
import json
import itertools
from collections import Counter
from copy import deepcopy

import ast
import pandas as pd

In [2]:
# API_KEY = "fc5ce3f2-8730-4348-b7b8-e25c07b640d1"  
# API_KEY = "091c4d2e-d45f-4529-b52c-748f74445ab4"
# API_KEY = "e8f39fa1-725e-4836-a395-deb32b98f6a7"
# API_KEY = "5762e390-5d48-41c4-876f-be60bc35e175"
# API_KEY = "c603d149-47e5-489f-9c3b-17db957d9f33"
BASE_URL = "https://content.guardianapis.com/search"

## Collect data

### Sections collection in json (we do not want to make api call every time)

In [3]:
# URL = "https://content.guardianapis.com/sections"

# params = {
#     "api-key": API_KEY
# }

# response = requests.get(URL, params=params)

# sections_lst = response.json()["response"]["results"]

# main_sections_dct = {}

# for el in sections_lst:
#     editions_ids = []
#     for edition in el["editions"]:
#         editions_ids.append(edition["id"])
        
#     main_sections_dct[el["id"]] = editions_ids

# main_sections_dct

# with open("main_sections_dct.json", "w", encoding="utf-8") as f:
    # json.dump(main_sections_dct, f, ensure_ascii=False)

# section_id_to_name_dct = {}
# for el in sections_lst:
#     section_id_to_name_dct[el["id"]] = el["webTitle"]
#     for edition in el["editions"]:
#         section_id_to_name_dct[edition["id"]] = edition["webTitle"]

# with open("section_id_to_name_dct.json", "w", encoding="utf-8") as f:
#     json.dump(section_id_to_name_dct, f, ensure_ascii=False)

In [4]:
with open("main_sections_dct.json", "r", encoding="utf-8") as f:
    main_sections_dct = json.load(f)

with open("section_id_to_name_dct.json", "r", encoding="utf-8") as f:
    section_id_to_name_dct = json.load(f)

In [5]:
main_sections_lst = list(main_sections_dct.keys())

main_sections_lst_all_editions = []
for _, v in main_sections_dct.items():
    main_sections_lst_all_editions += v

### Function to get one page

In [6]:
# page size - how many articles with return api
def fetch_page(page, page_size=50, section=None):
    params = {
        "api-key": API_KEY,
        "page": page,
        "page-size": page_size,
        "show-fields": "bodyText",
        "show-tags": "keyword",
        "order-by": "newest",
    }
    
    if section:
        params["section"] = section

    response = requests.get(BASE_URL, params=params)

    # if 4xx or 5xx error we raise HTTP error
    # response.raise_for_status()
    data = response.json()

    if "response" not in data:
        return None

    if data["response"]["status"] == "error":
        return None 
    
    return response.json()["response"]

### Parse one text function

In [7]:
def parse_article(item):
    tags = [t["id"] for t in item.get("tags", [])]
    tags_sections_ids = [t.get("sectionId") for t in item.get("tags", [])]
    tags_sections_names = [t.get("sectionName") for t in item.get("tags", [])]

    
    body_text = item.get("fields", {}).get("bodyText", "")

    return {
        "id": item["id"],
        "url": item["webUrl"],
        "title": item["webTitle"],
        "section_id": item["sectionId"],
        "section_name": item["sectionName"],
        "published_date": item["webPublicationDate"],
        "body_text": body_text,
        "tags": tags,
        "tags_sections_ids": tags_sections_ids,
        "tags_sections_names": tags_sections_names,
    }

### Collect pages

LIMIT: 5K a day

In [8]:
def collect_articles(sections, pages_per_section: int = 50, page_size: int = 100, start_page: int = 1):
    all_articles = []

    for section in sections:
        for page in range(start_page, start_page + pages_per_section):
            data = fetch_page(page=page, page_size=page_size, section=section)
            if data is None:
                break 
            for item in data["results"]:
                all_articles.append(parse_article(item))
            time.sleep(0.2)  

    return all_articles

In [9]:
# # chosen_sections = ['books', 'business', 'cities', 'community', 'culture']
# len(main_sections_lst)

# chosen_sections = main_sections_lst[-40:] # 30

In [10]:
# all_articles = collect_articles(chosen_sections, pages_per_section=70, start_page=1,)

In [11]:
# # all_articles

# file_paths = [
#     "data_collected8.csv", 
#     "data_collected7.csv",
#     "data_collected6.csv",
#     "data_collected4.csv",
#     "data_collected3.csv",
#     "data_collected2.csv"
# ]

# df_collected = pd.read_csv("data_collected.csv")

# for path in file_paths:
#     cur_df = pd.read_csv(path)

#     df_collected = pd.concat([df_collected, cur_df])

# # df_collected = df_collected.drop(columns=["Unnamed:"])

# df_collected_filtered = df_collected.drop(columns=["Unnamed: 0"])

# df_collected_clean = df_collected_filtered.drop_duplicates().reset_index()

# df_collected_clean.to_csv("data_clean_final.csv")

## Data preparation: filtering and etc.

In [12]:
df_final_clean = pd.read_csv("data_clean_final.csv").drop(columns=["Unnamed: 0", "index"]).reset_index()

### We need to get rid of texts with too rare sections

In [13]:
section_id_counts = df_final_clean["section_id"].value_counts()

MIN_NUM_EXAMPLES_FOR_SECTION = 2000

sections_to_save = section_id_counts[section_id_counts >= MIN_NUM_EXAMPLES_FOR_SECTION].index.tolist()

df_final_clean_filtered = df_final_clean[df_final_clean["section_id"].isin(sections_to_save)].copy()

print(f"Number of rows in DataFrame BEFORE filtering on section_id: {df_final_clean.shape[0]}")
print(f"Number of rows in DataFrame AFTER filtering on section_id: {df_final_clean_filtered.shape[0]}")

Number of rows in DataFrame BEFORE filtering on section_id: 267088
Number of rows in DataFrame AFTER filtering on section_id: 246883


### TAGS FILTERING: Approach one: get rid of tags that are same as section_id + save only top N most popular

In [14]:
# Function to get rid of tags which are like 'section/section'
def filter_same_tag_as_section(row):
    new_tags = []

    tags_list = row["tags"]
    if isinstance(tags_list, str):
        tags_list = ast.literal_eval(tags_list)

    for tag in tags_list:
        if "/" in tag:
            parts = tag.split("/")
            
            if len(parts) == 2 and parts[0] == parts[1] and parts[0] == row["section_id"]:
                continue
            else:
                new_tags.append(tag)
        else:
            new_tags.append(tag)

    return new_tags

tags_without_same_as_section = df_final_clean_filtered.apply(lambda x: filter_same_tag_as_section(x), axis=1)

df_final_clean_filtered["filtered_tags"] = tags_without_same_as_section

Look on the tags statistic and on the most popular tags

In [15]:
all_tags_lst = list(itertools.chain.from_iterable(df_final_clean_filtered["filtered_tags"]))

all_tags_set = set(all_tags_lst)

print(f"Total number of tags (not unique): {len(all_tags_lst)}")
print(f"Total number of unique tags: {len(all_tags_set)}")

Total number of tags (not unique): 1289102
Total number of unique tags: 15077


In [16]:
tag_counts = Counter(all_tags_lst)

N_TOP_TAGS = 50

top_n_tags = tag_counts.most_common(N_TOP_TAGS)

tags_to_filter = [
    'world/coronavirus-outbreak', 
    'politics/keir-starmer', 
    'uk/london', 
    'us-news/trump-administration',
    'society/nhs',
    'us-news/donaldtrump',
    'world/world',
    # 'culture/culture'
    # 'society/society'
    'politics/labour',
    'world/americas',
    'society/communities',
    'society/housing',
    'books/childrens-books-8-12-years',
]

final_saved_tags_set = set([el[0] for el in top_n_tags if el[0] not in tags_to_filter])

print(f"Total number of tags that we will use: {len(final_saved_tags_set)}")

new_filtered_tags = df_final_clean_filtered["filtered_tags"].apply(
    lambda tags: list(set(tags) & final_saved_tags_set))

df_final_clean_filtered["new_filtered_tags"] = new_filtered_tags

df_final_clean_filtered = df_final_clean_filtered[
    df_final_clean_filtered["new_filtered_tags"].apply(len) > 0
].reset_index(drop=True)

# Get rid of too short text 
df_final_clean_filtered = df_final_clean_filtered[df_final_clean_filtered["body_text"].str.len() > 200].reset_index(drop=True)

print("Shape of DataFrame after tags filtering and short texts filtering", df_final_clean_filtered.shape)

Total number of tags that we will use: 40
Shape of DataFrame after tags filtering and short texts filtering (180283, 13)


### Save only max N texts

In [17]:
df_final_clean_filtered = df_final_clean_filtered.drop(columns=["tags", "filtered_tags"]).rename(columns={"new_filtered_tags": "tags"})

# Rank texts based on the length and save maximum 3000 texts for each section
df_final_clean_filtered["text_len"] = df_final_clean_filtered['body_text'].str.len()

rank = (
    df_final_clean_filtered.groupby('section_id')['text_len']
    .rank(ascending=True, method='first')
    .astype(int)
)

df_final_clean_filtered = df_final_clean_filtered[rank <= 3000]

In [18]:
df_final_clean_filtered.shape

(111825, 12)

### Save file

In [19]:
df_final_clean_filtered.to_csv("clean_data_for_section_tags_modelling/df_final_clean_filtered_new_new_v1.csv")